# Notebook 02 — LSTM Forecaster (Baseline)
**Syllabus Week 9 | CLO 3: Sequence Models**

## Problem Statement
Train a 2-layer LSTM to forecast the next 24 hours of `temperature_2m` given
a 48-hour window of 7 weather features. This serves as the **baseline** for
comparison against the from-scratch Transformer (Notebook 03).

**Same task, same data, same seeds** as the Transformer — apples-to-apples.

| Hyperparameter | Value |
|---|---|
| Input | (B, 48, 7) — 48h window, 7 features |
| Output | (B, 24) — next 24h temperature |
| Layers | 2 |
| Hidden size | 64 |
| Bidirectional | False (causal — cannot see future) |
| Dropout | 0.1 |
| Params | ~36K |

## Section 2 — Math Derivation

### LSTM Cell Equations
At each timestep $t$, given input $x_t \in \mathbb{R}^F$ and hidden state $h_{t-1} \in \mathbb{R}^H$:

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \quad \text{(cell candidate)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cell state)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(hidden state)}$$

### Forecast Head
After processing all 48 timesteps, the last layer's final hidden state $h_{T}^{(L)}$ is projected:
$$\hat{y} = W_{\text{head}} \, h_T^{(L)} + b_{\text{head}}, \quad \hat{y} \in \mathbb{R}^{24}$$

### Training Loss
Mean Squared Error over the 24-step horizon:
$$\mathcal{L} = \frac{1}{24} \sum_{h=1}^{24} (\hat{y}_h - y_h)^2$$

Gradients flow through BPTT (Backpropagation Through Time) for all 48 steps.
Gradient clipping at norm 1.0 prevents exploding gradients.

In [ ]:
# Section 3 — Data Loading
# TODO: run after python -m ml.data_pipeline
import sys; sys.path.insert(0, '..')
import torch
from torch.utils.data import DataLoader

# from ml.train.train_transformer import load_data   # reused from transformer module
# train_ds, val_ds, test_ds = load_data()
# print(f'train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,} windows')
# X_sample, y_sample = train_ds[0]
# print(f'X shape: {X_sample.shape}')  # (48, 7)
# print(f'y shape: {y_sample.shape}')  # (24,)

In [ ]:
# Section 4 — Model Definition
# TODO: run after deps installed
import torch.nn as nn

# from ml.train.train_lstm import WeatherLSTM
# model = WeatherLSTM()
# n_params = sum(p.numel() for p in model.parameters())
# print(f'Params: {n_params:,}')  # expect ~36K

# Tensor shapes through the model:
# x            : (B=64, T=48, F=7)
# lstm output  : (B, T, H=64)         -- all hidden states
# h_n          : (n_layers=2, B, H=64) -- final hidden per layer
# h_n[-1]      : (B, H=64)             -- last layer's final hidden
# head output  : (B, 24)               -- forecast

In [ ]:
# Section 5 — Training Loop (run python -m ml.train.train_lstm instead)
# Results captured here after training:

# TODO: paste epoch log here after training
# epoch 01  train_mse=X.XXXX  val_mse=X.XXXX
# ...
# Test MSE=X.XXXX  MAE=X.XXXX

# from pathlib import Path
# import matplotlib.pyplot as plt, numpy as np
# curve = np.load('../models/lstm/train_curve.npy', allow_pickle=True).item()
# # Or just display the saved PNG:
# from IPython.display import Image
# Image('../models/lstm/train_curve.png')

## Section 6 — Architecture Diagram

```
Input (B, 48, 7)
      │
      ▼
 ┌─────────────────────────────────┐
 │  LSTM Layer 1  (hidden=64)      │
 │  dropout=0.1 between layers     │
 │  LSTM Layer 2  (hidden=64)      │
 └─────────────────────────────────┘
      │  h_n[-1]  (B, 64)
      ▼
 Linear(64 → 24)
      │
      ▼
 Output (B, 24)  ← next 24h temperature forecast
```

**Why `h_n[-1]` not `out[:, -1]`?**
`h_n[-1]` is the final hidden state of the *last* LSTM layer — it has seen
all 48 timesteps and distilled them through both layers. `out[:, -1]` is
the output of the last *timestep* from layer 2 only — equivalent here, but
`h_n[-1]` is cleaner and generalises to bidirectional models.

In [ ]:
# Section 7 — Evaluation
# TODO: fill in after training
from IPython.display import Image

# Display training curve
# Image('../models/lstm/train_curve.png')

# TODO: plot first 5 test predictions vs ground truth
# import torch
# model.load_state_dict(torch.load('../models/lstm/model.pt', map_location='cpu'))
# model.eval()
# X_test, y_test = test_ds[:5]  # (5, 48, 7), (5, 24)
# with torch.no_grad():
#     preds = model(X_test)  # (5, 24)
# fig, axes = plt.subplots(1, 5, figsize=(20, 3))
# for i, ax in enumerate(axes):
#     ax.plot(y_test[i].numpy(), label='true')
#     ax.plot(preds[i].numpy(), label='pred', linestyle='--')
#     ax.set_title(f'Sample {i+1}')
#     ax.legend()
# plt.suptitle('LSTM: 24h Temperature Forecast vs Ground Truth (standardised)')
# plt.tight_layout(); plt.show()

# Metrics to record:
# Test MSE  = TODO
# Test MAE  = TODO

## Section 8 — Comparison vs Baseline & Reflection

### LSTM vs Transformer (to fill after both training runs)
| Model | Params | Test MSE | Test MAE | Train time |
|---|---|---|---|---|
| LSTM (this) | ~36K | TODO | TODO | TODO min |
| Transformer | ~400K | TODO | TODO | TODO min |

### Expected narrative
The LSTM baseline is ~11× smaller in parameters. If the Transformer achieves
meaningfully lower MSE, it justifies the attention mechanism's overhead.
If scores are comparable, the LSTM is the better engineering choice at this scale.

### Trade-offs
| Aspect | LSTM | Transformer |
|---|---|---|
| Inductive bias | Sequential, causal by construction | Explicit positional encoding needed |
| Parallelism | Sequential — slow to train | Fully parallel — faster per epoch |
| Long-range dependencies | Weakens with sequence length | Uniform attention over all positions |
| Interpretability | Hidden state opaque | Attention weights visualisable |
| CPU inference latency | Low (36K params) | Higher (400K params) |

### Limitations
- Single-step forecast head — no autoregressive decoding for longer horizons.
- `bidirectional=False` restricts to causal use; a bidirectional variant
  could serve offline analysis but not real-time inference.